# Fetch Market Data

## Overview

This notebook shows how to download raw historical trade data with the `fetch_market_data` module and persist it under `data/raw/alpaca/`.

Key points:
- `fetch_alpaca_historical_trades` requests historical trades from Alpaca and returns them as a pandas `DataFrame`.
- `save_alpaca_historical_trades` downloads the same data and saves it as a parquet file for later notebooks.

In [1]:
from pathlib import Path
import sys
from datetime import datetime, timezone

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists())
sys.path.insert(0, str(root))

In [2]:
from src.data_preprocessing.fetch_market_data import (
    fetch_alpaca_historical_trades,
    save_alpaca_historical_trades,
)

## Configure Request

This section defines the request parameters before calling the API.

Key points:
- `asset_class` selects the Alpaca endpoint family.
- `symbols` controls which instruments are requested.
- `start` and `end` define the UTC request window.
- `stock_feed` and `crypto_feed` choose the raw trade feed for the matching asset class.
- `output_path=None` lets the save helper use the module's default raw-data location and filename pattern.

In [3]:
asset_class = "stock"
symbols = ["AAPL"]
start = datetime(2026, 1, 1, 0, 0, tzinfo=timezone.utc)
end = datetime(2026, 1, 31, 0, 0, tzinfo=timezone.utc)
stock_feed = "iex"
crypto_feed = "us"
output_path = None

## Fetch Trades Into Memory

This section calls `fetch_alpaca_historical_trades` and returns a pandas `DataFrame` without writing to disk.

Key points:
- The returned frame is useful for quick inspection before saving raw data.
- `timestamp` is the trade time.
- `symbol` is the instrument identifier.
- `price` and `size` describe the executed trade.
- `id` and `exchange` are source-specific trade metadata from Alpaca.

In [4]:
trades = fetch_alpaca_historical_trades(
    symbols=symbols,
    start=start,
    end=end,
    asset_class=asset_class,
    stock_feed=stock_feed,
    crypto_feed=crypto_feed,
)
trades.head()

,timestamp,symbol,price,size,id,exchange
0,2026-01-02 13:27:19.847471+00:00,AAPL,273.23,100.0,1,V
1,2026-01-02 14:08:22.630628+00:00,AAPL,272.96,2.0,2,V
2,2026-01-02 14:08:32.020709+00:00,AAPL,272.91,20.0,3,V
3,2026-01-02 14:08:32.020733+00:00,AAPL,272.91,5.0,4,V
4,2026-01-02 14:30:00.046988+00:00,AAPL,272.04,1.0,5,V


## Save Raw Dataset

This section calls `save_alpaca_historical_trades` to persist the same request result as a parquet file.

Key points:
- When `output_path` is left as `None`, the helper uses the project's default raw-data location.
- The returned `Path` tells you exactly where the file was written.
- That saved parquet path can be reused in downstream notebooks.

In [5]:
saved_path = save_alpaca_historical_trades(
    symbols=symbols,
    start=start,
    end=end,
    asset_class=asset_class,
    output_path=output_path,
    stock_feed=stock_feed,
    crypto_feed=crypto_feed,
)
saved_path

PosixPath('/Users/kwonjunhyuk9/Documents/multi-asset-investing/data/raw/alpaca/stock_aapl_trades_20260101T000000Z_20260131T000000Z.parquet')